In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
from QPC_Optimizer import *
from state_measure_extension import *
from state_measure_extension import _prob_fully_entangled
from scipy.stats import unitary_group

### Parameter

In [ ]:
use_prev_matrix = False
qpc_parameter = (1,1)
ancilla = [1]
num_photon = sum(ancilla) + 2
sigmas = (0.05, 0.01)
polynom_orders = [8, 16]
sample_sizes = (40, 10)
forced = True
bound = _prob_fully_entangled(sum(ancilla))+1e-4
it = 0
weight = 0
measure_type = 7
number_of_iterations = len(sample_sizes)

# file names and path for optimized unitary and iteration results

path = 'unitaries/Xrot/'

fname_coeffs = [path + f'DRa={ancilla}_{i+1}.npy' for i in range(number_of_iterations)]
fname_final_matrix = path + f'DRa={ancilla}_matrix.npy'
fname_coeff_order = path + f'DRa={ancilla}_order.npy'

### Code for optimized unitary


In [ ]:
sample_size1 = 0
print(ancilla)

#clearing old data
c = True
if c:
    for fname in fname_coeffs:
        open(fname, 'w').close()

#generate initial matrix
DR = MatrixQPCFusion_with_arbitrary_state_measure(qpc_parameter,())
initial_matrix = unitary_group.rvs(4 * qpc_parameter[0] * qpc_parameter[1] + len(ancilla))


if use_prev_matrix:
    best_matrix = read_all_data_from_file(fname_final_matrix)
    psi = setting_up_Simulator_with_unitary_xrot(best_matrix[0], qpc_parameter, ancilla)
    initial_matrix = psi.give_current_lo_unitary()

#optional Parameters
decimals=3

#checking that parameters are valid
n,m = qpc_parameter
number_of_modes = 4 * n * m + len(ancilla)
if not number_of_iterations == len(sample_sizes) == len(sigmas) == len(fname_coeffs) == len(polynom_orders):
    raise ValueError
if np.shape(initial_matrix) != (number_of_modes, number_of_modes):
    raise ValueError

creation_operator = precompute_creation_operators(n,m, ancilla)
coeffs, arrangement_order = unitary_to_clements_coeffs(initial_matrix, decimals=decimals)

save_coeff_order(fname_coeff_order,arrangement_order)

#starting iteration
for iteration in range(number_of_iterations):
    print('iteration:',iteration + 1)

    while(True): # can be used to force a maximal success probability

        for _ in range(sample_sizes[iteration]):

            if iteration == 0: # for counting used samples of iteration 1
                sample_size1 += 1

            clements_coeffs = coeffs + np.random.normal(0, sigmas[iteration], len(coeffs))
            opt_coeff, opt_xrot_bellness = QPC_optimizer_probability_weighted_state_measure_of_all_states(clements_coeffs,
                                                                                   arrangement_order,
                                                                                   creation_operator,
                                                                                   qpc_parameter, ancilla,
                                                                                   polynom_orders[iteration], state_measure_type = measure_type)
            append_to_file(fname_coeffs[iteration], opt_coeff)
            append_to_file(fname_coeffs[iteration], opt_xrot_bellness)


        coeffs2 = select_best_coeffs_from_simulation_runs(fname_coeffs[iteration])
        U = unitary_from_clements(number_of_modes,coeffs2,arrangement_order)
        psi = setting_up_Simulator_with_unitary_xrot(U, qpc_parameter, ancilla)
        prob = 0

        if not forced:
            print('measure:',psi.xrot_Bellness_of_all_states(100, weight = weight))
            print('xrot_Bellness:',psi.xrot_Bellness_of_all_states(100, weight = 0))
            print('xrot_ZZness:',psi.xrot_Bellness_of_all_states(100, weight = 1))
            psi.select_bell_measurement_up_to_xrot_pattern(1e-3)
            prob = psi.probability_of_selected_events()
            print('prob of success:',prob)

        if forced:
            print('measure:',psi.xrot_Bellness_of_all_states(100, weight = weight))
            print('sample:', sample_size1)

        if iteration != 0 or (not forced):
            coeffs = coeffs2
            break
        
        else:
            # print('hi2')
            if psi.xrot_Bellness_of_all_states(100, weight = weight) > bound:
                print('found')
                coeffs = coeffs2
                break
            else:
                forced = True
                if c:
                    for fname in fname_coeffs:
                        open(fname, 'w').close()
                initial_matrix = unitary_group.rvs(4*qpc_parameter[0]*qpc_parameter[1]+len(ancilla))
                coeffs, arrangement_order = unitary_to_clements_coeffs(initial_matrix, decimals=decimals)
                save_coeff_order(fname_coeff_order,arrangement_order)
                

save_best_matrix_from_simulation_runs(fname_coeffs[-1], fname_final_matrix, qpc_parameter, ancilla)
best_matrix = read_all_data_from_file(fname_final_matrix)
psi = setting_up_Simulator_with_unitary_xrot(best_matrix[0],qpc_parameter, ancilla)
print('measure:',psi.xrot_Bellness_of_all_states(100))
psi.select_bell_measurement_up_to_xrot_pattern(1e-3)
print('prob of success:',psi.probability_of_selected_events())
prob_succ = psi.probability_of_selected_events()
#psi.select_partially_entangling_measurement_patterns(0.999)
#print(psi.probability_of_selected_events())

#print(best_matrix)
#print(initial_matrix)

best_matrix = read_all_data_from_file(fname_final_matrix)
psi = setting_up_Simulator_with_unitary_xrot(best_matrix[0],qpc_parameter, ancilla)
psi.choose_state_measure(6)
psi.select_ness_measurement(4,1e-3)
print('prob of ZZ-success:',psi.probability_of_selected_events())
prob_ZZ = psi.probability_of_selected_events()
print('prob of ZZ-success in failed',(prob_ZZ-prob_succ)/(1-prob_succ))

[2]
iteration: 1
measure: 2.1492817895035247e-18
sample: 40


KeyboardInterrupt: 

### Code for optimized unitary with ancilla photons for only one qubit

In [ ]:
#code for split

sample_size1 = 0
print(ancilla)

#clearing old data
c = True
if c:
    for fname in fname_coeffs:
        open(fname, 'w').close()

#generate initial matrix
initial_matrix = unitary_group.rvs(4 * qpc_parameter[0] * qpc_parameter[1] + len(ancilla) - 2)

if use_prev_matrix:
    #initial_matrix = unitary_to_higher_mode_number_unitary(read_all_data_from_file(f'unitaries/Xrot/DR_XRBSa={ancilla[:-2]}_matrix.npy')[0],2)
    best_matrix = read_all_data_from_file(fname_final_matrix)
    psi = setting_up_Simulator_with_unitary_xrot(best_matrix[0], qpc_parameter, ancilla)
    initial_matrix = psi.give_current_lo_unitary()

#optional Parameters
decimals=3

#checking that parameters are valid
n,m = qpc_parameter
number_of_modes = 4 * n * m + len(ancilla)
if not number_of_iterations == len(sample_sizes) == len(sigmas) == len(fname_coeffs) == len(polynom_orders):
    raise ValueError
if np.shape(initial_matrix) != (number_of_modes -2, number_of_modes -2):
    raise ValueError

# ma = read_all_data_from_file(f'unitaries/Xrot/DR_XRBSa={ancilla[:-2]}_matrix.npy')[0]

creation_operator = precompute_creation_operators(n,m, ancilla)
coeffs, arrangement_order = unitary_to_clements_coeffs(initial_matrix, decimals=decimals)

save_coeff_order(fname_coeff_order,arrangement_order)

for iteration in range(number_of_iterations):
    print('iteration:',iteration + 1)

    while(True): # can be used to force a maximal success probability

        for _ in range(sample_sizes[iteration]):

            if iteration == 0: # for counting used samples of iteration 1
                sample_size1 += 1

            clements_coeffs = coeffs + np.random.normal(0, sigmas[iteration], len(coeffs))
            opt_coeff, opt_xrot_bellness = QPC_optimizer_probability_weighted_state_measure_of_all_states(clements_coeffs,
                                                                                   arrangement_order,
                                                                                   creation_operator,
                                                                                   qpc_parameter, ancilla,
                                                                                   polynom_orders[iteration], state_measure_type = measure_type, num_photon = num_photon, weight = weight,
                                                                                   split = True)
            append_to_file(fname_coeffs[iteration], opt_coeff)
            append_to_file(fname_coeffs[iteration], opt_xrot_bellness)

        coeffs2 = select_best_coeffs_from_simulation_runs(fname_coeffs[iteration])
        U = unitary_from_clements(number_of_modes - 2, coeffs2, arrangement_order)
        U = add_init_to_split(U)
        psi = setting_up_Simulator_with_unitary_xrot(U, qpc_parameter, ancilla)
        prob = 0

        if not forced:
            print('measure:',psi.xrot_Bellness_of_all_states(100, weight = weight))
            print('xrot_Bellness:',psi.xrot_Bellness_of_all_states(100, weight = 0))
            print('xrot_ZZness:',psi.xrot_Bellness_of_all_states(100, weight = 1))
            psi.select_bell_measurement_up_to_xrot_pattern(1e-3)
            prob = psi.probability_of_selected_events()
            print('prob of success:',prob)

        if forced:
            print('measure:',psi.xrot_Bellness_of_all_states(100, weight = weight))
            print('sample:', sample_size1)

        if iteration != 0 or (not forced):
            coeffs = coeffs2
            break
        
        else:
            # print('hi2')
            if psi.xrot_Bellness_of_all_states(100, weight = weight) > bound:
                print('found')
                coeffs = coeffs2
                break
            else:
                forced = True
                initial_matrix = unitary_group.rvs(4*qpc_parameter[0]*qpc_parameter[1]+len(ancilla) - 2)
                coeffs, arrangement_order = unitary_to_clements_coeffs(initial_matrix, decimals=decimals)
                save_coeff_order(fname_coeff_order,arrangement_order)
                


save_best_matrix_from_simulation_runs(fname_coeffs[-1], fname_final_matrix, qpc_parameter, ancilla[2:])
best_matrix = read_all_data_from_file(fname_final_matrix)
print(best_matrix[0])
psi = setting_up_Simulator_with_unitary_xrot(add_init_to_split(best_matrix[0]),qpc_parameter, ancilla)
print('measure:',psi.xrot_Bellness_of_all_states(100))
#print(psi.give_current_lo_unitary())
#print(np.abs(psi.give_current_lo_unitary()))
#print(np.angle(psi.give_current_lo_unitary())/np.pi)
psi.select_bell_measurement_up_to_xrot_pattern(1e-3)
print('prob of success:',psi.probability_of_selected_events())
prob_succ = psi.probability_of_selected_events()
#psi.select_partially_entangling_measurement_patterns(0.999)
#print(psi.probability_of_selected_events())

#print(best_matrix)
#print(initial_matrix)

best_matrix = read_all_data_from_file(fname_final_matrix)
psi = setting_up_Simulator_with_unitary_xrot(add_init_to_split(best_matrix[0]),qpc_parameter, ancilla)
psi.chose_ness_function(6)
psi.select_ness_measurement(4,1e-3)
print('prob of ZZ-success:',psi.probability_of_selected_events())
prob_ZZ = psi.probability_of_selected_events()
print('prob of ZZ-success in failed',(prob_ZZ-prob_succ)/(1-prob_succ))

[2]
iteration: 1


KeyboardInterrupt: 